# Fine-tuning LLM untuk Chatbot Hukum Indonesia

**Modul**: Pengembangan Generative AI berbasis LLM — Digdaya Hackathon BI

**Target**: Advanced (4 pts)

---

## Ringkasan Notebook

1. Load dataset `Ichsan2895/alpaca-gpt4-indonesian` (format Alpaca, Bahasa Indonesia)
2. Train/validation split (90/10)
3. Mapping dataset ke ChatML template (format native Qwen2.5)
4. Load `Qwen/Qwen2.5-7B-Instruct` dengan QLoRA 4-bit + double quantization
5. LoRA adapters pada komponen Multi-Head Attention
6. Dua eksperimen SFTTrainer dengan hyperparameter berbeda
7. Loss curve comparison
8. Merge LoRA (merged_16bit) + push ke HuggingFace Hub

> **Catatan**: Notebook ini didesain untuk Google Colab dengan GPU T4 (16GB VRAM).

In [1]:
# Clone repo ke Colab runtime (cukup sekali)
# !git clone https://github.com/RayhanLup1n/chatbot_rag_digdaya.git

In [2]:
# Masuk ke folder
import os
os.chdir("/content/chatbot_rag_digdaya")

In [3]:
%cd /content/chatbot_rag_digdaya
!git pull origin main

/content
From https://github.com/RayhanLup1n/chatbot_rag_digdaya
 * branch            main       -> FETCH_HEAD
Already up to date.


## 1. Instalasi Dependencies

In [4]:
# Install Unsloth dan dependencies (Colab/Kaggle compatible)
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub safetensors matplotlib

In [5]:
# [PENTING] Setelah cell instalasi di atas selesai:
# RESTART RUNTIME secara manual: Runtime → Restart runtime (Ctrl+M .)
# Setelah restart, SKIP 2 cell instalasi ini dan LANJUTKAN dari "2. Import Libraries"
# Cell ini hanya instruksi — tidak dijalankan setelah restart
print("\n" + "=" * 60)
print("INSTALASI SELESAI.")
print("SEKARANG: Runtime → Restart runtime (Ctrl+M .)")
print("Setelah restart, skip cell instalasi, lanjut ke IMPORT.")
print("=" * 60)


INSTALASI SELESAI.
SEKARANG: Runtime → Restart runtime (Ctrl+M .)
Setelah restart, skip cell instalasi, lanjut ke IMPORT.


## 2. Import Libraries

In [6]:
# [PENTING] unsloth HARUS di-import paling awal sebelum transformers/trl/peft
import os
os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import unsloth  # noqa: F401 — init Unsloth optimizations
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import matplotlib.pyplot as plt
import os
import re

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: GPU tidak terdeteksi! Pastikan Runtime → Change runtime type → T4 GPU")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


## 3. Load Dataset

Dataset: `Ichsan2895/alpaca-gpt4-indonesian` — format Alpaca (instruction, input, output) dalam Bahasa Indonesia.

In [7]:
# Load dataset dari HuggingFace
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(f"Jumlah total sample: {len(dataset)}")
print(f"Kolom: {dataset.column_names}")

Jumlah total sample: 49969
Kolom: ['Unnamed: 0', 'input', 'output']


In [8]:
# Tampilkan beberapa baris dataset MENTAH (sebelum mapping)
# Dataset Ichsan2895/alpaca-gpt4-indonesian kolom: ['Unnamed: 0', 'input', 'output']
# - Unnamed: 0 = row index (abaikan)
# - input = instruction + context (gabungan)
# - output = jawaban

print("=" * 80)
print("DATASET MENTAH — Sebelum Chat Template Mapping")
print(f"Kolom: {dataset.column_names}")
print("=" * 80)
for i in range(3):
    sample = dataset[i]
    print(f"--- Sample #{i+1} ---")
    print(f"instruction (input): {str(sample['input'])[:300]}...")
    print(f"output: {str(sample['output'])[:300]}...")
    print()

DATASET MENTAH — Sebelum Chat Template Mapping
Kolom: ['Unnamed: 0', 'input', 'output']
--- Sample #1 ---
instruction (input): Saranlah slogan untuk kampanye daur ulang
...
output: 1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."...

--- Sample #2 ---
instruction (input): Ciptakan cara unik dan lucu untuk menggambarkan presiden negaramu.
...
output: Sebagai AI, saya tidak tergabung dalam satu negara dan karena itu tidak memiliki presiden tertentu yang harus digambarkan. Namun, saya dapat membantu Anda menciptakan deskripsi unik dan lucu tentang presiden negara Anda jika Anda memberikan informasi lebih lanjut!...

--- Sample #3 ---
instruction (input): Diberikan deskripsi tentang peristiwa pribadi, tuliskan satu bait sajak haiku.
Berlayar di laut d

## 4. Train/Validation Split

Split dataset 90% training, 10% validation untuk memenuhi kriteria Skilled & Advanced.

In [9]:
# Train/validation split (90/10)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

Training samples: 44972
Validation samples: 4997


## 5. Chat Template Mapping

Mapping dataset ke **ChatML format** (format native Qwen2.5).

Format ChatML:
```
<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{instruction + input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>
```

Token spesial ChatML:
- `<|im_start|>` — penanda awal pesan (role)
- `<|im_end|>` — penanda akhir pesan
- Role tokens: `system`, `user`, `assistant`

In [10]:
# Define system prompt dalam Bahasa Indonesia
SYSTEM_PROMPT = """Anda adalah asisten AI yang membantu menjawab pertanyaan dalam Bahasa Indonesia. Berikan jawaban yang akurat, informatif, dan sopan."""

def format_chatml(example):
    """
    Mapping function: mengubah row dataset Alpaca ke format ChatML.

    Dataset Ichsan2895/alpaca-gpt4-indonesian kolom:
    - input: berisi instruction/pertanyaan
    - output: berisi jawaban

    Format ChatML:
    <|im_start|>system
    {system}<|im_end|>
    <|im_start|>user
    {input}<|im_end|>
    <|im_start|>assistant
    {output}<|im_end|>
    """
    # input = instruction dalam dataset ini
    user_message = example.get("input", "") or ""
    assistant_message = example.get("output", "") or ""

    # Bangun teks lengkap format ChatML
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant_message}<|im_end|>"
    )

    return {"text": text}

# Terapkan mapping ke train dan eval dataset
train_dataset_mapped = train_dataset.map(format_chatml)
eval_dataset_mapped = eval_dataset.map(format_chatml)

print("Mapping selesai. Kolom baru: 'text' ditambahkan.")

Mapping selesai. Kolom baru: 'text' ditambahkan.


In [11]:
# Tampilkan 1 baris dataset yang SUDAH di-mapping (lengkap dengan token spesial)
print("=" * 80)
print("DATASET SUDAH DI-MAPPING — Format ChatML dengan Token Spesial")
print("=" * 80)
print()

sample_mapped = train_dataset_mapped[0]
print(sample_mapped["text"])

print()
print("=" * 80)
print("Token spesial yang digunakan:")
print("  <|im_start|>  — penanda awal role (system/user/assistant)")
print("  <|im_end|>    — penanda akhir pesan")
print("  system, user, assistant — nama role")
print("=" * 80)

DATASET SUDAH DI-MAPPING — Format ChatML dengan Token Spesial

<|im_start|>system
Anda adalah asisten AI yang membantu menjawab pertanyaan dalam Bahasa Indonesia. Berikan jawaban yang akurat, informatif, dan sopan.<|im_end|>
<|im_start|>user
Deskripsikan sebuah karakter dalam buku "Dracula".
<|im_end|>
<|im_start|>assistant
Salah satu karakter di buku "Dracula" adalah Count Dracula sendiri. Dia adalah karakter utama dan antagonis utama dari novel. Dracula adalah vampir berusia berabad-abad, yang tinggal di kastil yang membusuk di Pegunungan Karpatia di Transylvania. Dia digambarkan sebagai sosok yang tinggi dan kurus, dengan kulit pucat dan fitur tajam. Mata Dracula bisa jadi hipnotis atau menakutkan, tergantung pada mood-nya, dan dia digambarkan memiliki aura pesona dan ancaman. Meskipun memiliki perilaku aristokrat, Dracula adalah predator jahat yang menghisap darah orang hidup untuk mempertahankan kekuatan dan keabadiannya. Tujuannya adalah menyebar kutukan vampirnya ke seluruh duni

## 6. Load Model dengan QLoRA 4-bit + Double Quantization

Menggunakan `Qwen/Qwen2.5-7B-Instruct` sebagai base model. QLoRA configuration:
- **4-bit quantization** dengan `bitsandbytes` (`load_in_4bit=True`)
- **Double quantization** (`bnb_4bit_use_double_quant=True`) — quantisasi ulang konstanta quantisasi untuk mengurangi memory footprint lebih lanjut
- **NF4 datatype** (`bnb_4bit_quant_type="nf4"`) — NormalFloat4, format optimal untuk weight terdistribusi normal

In [12]:
# Model configuration\nmodel_name = \"Qwen/Qwen2.5-7B-Instruct\"\nMAX_TRAIN_LENGTH = 384\nMAX_EVAL_SAMPLES = 500\nmax_seq_length = MAX_TRAIN_LENGTH\nprint(f\"Base model: {model_name}\")\nprint(f\"Max sequence length: {MAX_TRAIN_LENGTH}\")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-7B-Instruct
Quantization: 4-bit NF4 + Double Quantization (QLoRA)
Max sequence length: 2048


In [13]:
print(\"Quantization verification runs per experiment after model load.\")

Quantization Config:
  bnb_4bit_compute_dtype: torch.float16
  bnb_4bit_quant_type: nf4
  bnb_4bit_use_double_quant: True
  llm_int8_enable_fp32_cpu_offload: False
  llm_int8_has_fp16_weight: False
  llm_int8_skip_modules: None
  llm_int8_threshold: 6.0
  load_in_4bit: True
  load_in_8bit: False
  quant_method: bitsandbytes


## 7. LoRA Configuration

LoRA adapters didefinisikan pada **seluruh komponen Multi-Head Attention (MHA)**:
- `q_proj` — Query projection
- `k_proj` — Key projection
- `v_proj` — Value projection
- `o_proj` — Output projection

Ini memastikan LoRA mencakup setidaknya satu komponen komputasi utama (MHA) secara penuh.

In [14]:
# Definisikan LoRA adapters pada komponen Multi-Head Attention
# Parameter lora_r dan lora_alpha akan divariasikan per eksperimen

def apply_lora(model, r=16, lora_alpha=16, lora_dropout=0.0):
    """
    Terapkan LoRA adapters pada model.

    Target modules: q_proj, k_proj, v_proj, o_proj (seluruh MHA)
    Juga menyertakan gate_proj, up_proj, down_proj (FFN) untuk coverage lebih luas
    """
    model = FastLanguageModel.get_peft_model(
        model,
        r=r,  # LoRA rank
        lora_alpha=lora_alpha,  # Scaling factor
        lora_dropout=lora_dropout,  # Regularization
        target_modules=[
            # Multi-Head Attention (KOMPONEN UTAMA - wajib)
            "q_proj", "k_proj", "v_proj", "o_proj",
            # Feed-Forward Network (tambahan untuk coverage lebih baik)
            "gate_proj", "up_proj", "down_proj",
        ],
        use_gradient_checkpointing="unsloth",  # Memory-efficient
        random_state=42,
        use_rslora=False,  # Standard LoRA
        loftq_config=None,
    )
    return model

print("Fungsi apply_lora() siap digunakan.")
print("Target modules MHA: q_proj, k_proj, v_proj, o_proj")
print("Target modules FFN: gate_proj, up_proj, down_proj")

Fungsi apply_lora() siap digunakan.
Target modules MHA: q_proj, k_proj, v_proj, o_proj
Target modules FFN: gate_proj, up_proj, down_proj


In [15]:
print(\"ChatML tokenizer will be initialized after each model load.\")

Tokenizer updated dengan ChatML template.


## 8. Eksperimen 1 — Default Hyperparameters

| Parameter | Value |
|-----------|-------|
| learning_rate | 2e-4 |
| per_device_train_batch_size | 4 |
| lora_r | 16 |
| lora_alpha | 16 |
| gradient_accumulation_steps | 4 |
| max_steps | 800 |
| evaluation_strategy | steps |
| eval_steps | 100 |
| logging_steps | 50 |
| lr_scheduler_type | cosine |

In [16]:
# Load model ulang untuk eksperimen 1
model_exp1, tokenizer_exp1 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Setup tokenizer ChatML
tokenizer_exp1 = get_chat_template(
    tokenizer_exp1,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply LoRA — Experiment 1 hyperparameters
# LoRA pada seluruh MHA: q_proj, k_proj, v_proj, o_proj
model_exp1 = FastLanguageModel.get_peft_model(
    model_exp1,
    r=16,  # LoRA rank — Experiment 1
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model Experiment 1 siap.")
print(f"LoRA rank (r): 16, lora_alpha: 16")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model Experiment 1 siap.
LoRA rank (r): 16, lora_alpha: 16


In [17]:
# Training Arguments — Experiment 1
training_args_exp1 = TrainingArguments(
    output_dir="./outputs/exp1",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,  # Learning rate lebih tinggi
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_steps=800,

    # [Skilled] Evaluation strategy
    eval_strategy="steps",
    eval_steps=400,
    save_strategy="steps",
    save_steps=400,

    # [Skilled] Logging
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,

    # Optimization
    optim="adamw_8bit",  # 8-bit AdamW untuk hemat VRAM
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Checkpointing
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Reporting
    report_to="none",  # Disable wandb/tensorboard untuk simplifikasi
    save_total_limit=2,

    # Seed
    seed=42,
)

print("TrainingArguments Experiment 1:")
print(f"  learning_rate: {training_args_exp1.learning_rate}")
print(f"  batch_size: {training_args_exp1.per_device_train_batch_size}")
print(f"  max_steps: {training_args_exp1.max_steps}")
print(f"  evaluation_strategy: {training_args_exp1.eval_strategy}")
print(f"  eval_steps: {training_args_exp1.eval_steps}")
print(f"  logging_steps: {training_args_exp1.logging_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments Experiment 1:
  learning_rate: 0.0002
  batch_size: 4
  max_steps: 800
  evaluation_strategy: IntervalStrategy.STEPS
  eval_steps: 100
  logging_steps: 50


In [18]:
# SFTTrainer — Experiment 1
trainer_exp1 = SFTTrainer(
    model=model_exp1,
    tokenizer=tokenizer_exp1,
    args=training_args_exp1,
    train_dataset=train_dataset_mapped,
    eval_dataset=eval_dataset_mapped,  # [Skilled] Validation dataset
    dataset_text_field="text",  # Kolom hasil mapping ChatML
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Tidak packing agar setiap sample sequence terpisah
)

print("SFTTrainer Experiment 1 siap.")
print(f"Training samples: {len(train_dataset_mapped)}")
print(f"Eval samples: {len(eval_dataset_mapped)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/44972 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4997 [00:00<?, ? examples/s]

SFTTrainer Experiment 1 siap.
Training samples: 44972
Eval samples: 4997


In [19]:
# Jalankan training — Experiment 1
import torch
import gc
from transformers import TrainingArguments

# Bersihkan memory cache secara menyeluruh
gc.collect()
torch.cuda.empty_cache()

# Update arguments untuk stabilitas tinggi pada T4 GPU
# Menggunakan standard TrainingArguments karena FastTrainingArguments mungkin tidak tersedia
trainer_exp1.args = TrainingArguments(
    output_dir="./outputs/exp1",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8, # Akumulasi tinggi untuk stabilitas
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_steps=800,
    eval_strategy="steps",
    eval_steps=400,
    save_strategy="steps",
    save_steps=400,
    logging_steps=50,
    optim="adamw_8bit",
    fp16=True,
    bf16=False,
    weight_decay=0.01,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=42,
)

print("=" * 60)
print("MEMULAI TRAINING — EXPERIMENT 1 (Standard Args + Memory Optimized)")
print(f"  Model: {model_name}")
print(f"  Batch Size: 2 | Accumulation: 8")
print("=" * 60)

try:
    trainer_exp1.train()
except Exception as e:
    print(f"\nError detected: {e}")
    print("SARAN: Jika muncul 'Illegal Memory Access' atau 'OOM', silakan lakukan 'Restart Session' di menu Runtime, lalu jalankan cell dari awal.")

# Simpan log training
if hasattr(trainer_exp1.state, 'log_history'):
    log_history_exp1 = trainer_exp1.state.log_history
    print(f"\nTraining selesai. Total log entries: {len(log_history_exp1)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


MEMULAI TRAINING — EXPERIMENT 1 (Standard Args + Memory Optimized)
  Model: Qwen/Qwen2.5-7B-Instruct
  Batch Size: 2 | Accumulation: 8

Error detected: 'AcceleratorState' object has no attribute 'is_fsdp2'
SARAN: Jika muncul 'Illegal Memory Access' atau 'OOM', silakan lakukan 'Restart Session' di menu Runtime, lalu jalankan cell dari awal.

Training selesai. Total log entries: 0


## 9. Eksperimen 2 — Alternative Hyperparameters

| Parameter | Value | Perubahan dari Exp 1 |
|-----------|-------|---------------------|
| learning_rate | 5e-5 | Lebih rendah (4x) |
| per_device_train_batch_size | 2 | Lebih kecil |
| lora_r | 32 | Lebih besar (2x) |
| lora_alpha | 64 | Lebih besar (4x) |
| gradient_accumulation_steps | 8 | Lebih besar (2x) |
| max_steps | 800 | Sama |
| evaluation_strategy | steps | Sama |
| eval_steps | 100 | Sama |
| logging_steps | 50 | Sama |

**Rasional**: Learning rate lebih rendah dengan LoRA rank lebih tinggi untuk mencari keseimbangan antara stabilitas training dan kapasitas adaptasi.

In [20]:
# Hapus model experiment 1 dari memory
import gc
del model_exp1, trainer_exp1
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

Memory cleared.


In [21]:
# Load model ulang untuk eksperimen 2
max_seq_length = 384

model_exp2, tokenizer_exp2 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

tokenizer_exp2 = get_chat_template(
    tokenizer_exp2,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply LoRA — Experiment 2 hyperparameters
# LoRA pada seluruh MHA: q_proj, k_proj, v_proj, o_proj
model_exp2 = FastLanguageModel.get_peft_model(
    model_exp2,
    r=16,  # LoRA rank — Experiment 2 (2x Experiment 1)
    lora_alpha=32,  # 4x Experiment 1
    lora_dropout=0.05,  # Dropout kecil untuk regularisasi
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model_exp2.config.use_cache = False

print("Model Experiment 2 siap.")
print(f"LoRA rank (r): 16, lora_alpha: 32, lora_dropout: 0.05")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model Experiment 2 siap.
LoRA rank (r): 32, lora_alpha: 64, lora_dropout: 0.05


In [22]:
# Training Arguments — Experiment 2
# training_args_exp2 = TrainingArguments(
#     output_dir="./outputs/exp2",
#     per_device_train_batch_size=2,  # Lebih kecil dari Exp 1
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8,  # Lebih besar (efektif batch size tetap 16)
#     learning_rate=5e-5,  # 4x lebih rendah dari Exp 1
#     lr_scheduler_type="cosine_with_restarts",  # Cosine with warm restarts
#     warmup_ratio=0.1,
#     max_steps=800,

#     # [Skilled] Evaluation strategy
#     eval_strategy="steps",
#     eval_steps=400,
#     save_strategy="steps",
#     save_steps=400,

#     # [Skilled] Logging
#     logging_strategy="steps",
#     logging_steps=50,
#     logging_first_step=True,

#     # Optimization
#     optim="adamw_8bit",
#     weight_decay=0.01,
#     max_grad_norm=1.0,

#     # Precision
#     fp16=not is_bfloat16_supported(),
#     bf16=is_bfloat16_supported(),

#     # Checkpointing
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,

#     # Reporting
#     report_to="none",
#     save_total_limit=2,

#     seed=123,  # Seed berbeda untuk eksperimen berbeda
# )

training_args_exp2 = TrainingArguments(
      output_dir="./outputs/exp2",
      per_device_train_batch_size=1,
      per_device_eval_batch_size=1,
      gradient_accumulation_steps=8,
      learning_rate=5e-5,
      lr_scheduler_type="cosine_with_restarts",
      warmup_ratio=0.05,
      max_steps=800,
      eval_strategy="steps",
      eval_steps=400,
      eval_accumulation_steps=1,
      save_strategy="steps",
      save_steps=400,
      logging_strategy="steps",
      logging_steps=50,
      logging_first_step=True,
      optim="adamw_8bit",
      weight_decay=0.01,
      max_grad_norm=0.5,
      fp16=True,
      bf16=False,
      tf32=False,
      gradient_checkpointing=True,
      gradient_checkpointing_kwargs={"use_reentrant": False},
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="eval_loss",
      greater_is_better=False,
      report_to="none",
      save_total_limit=1,
      dataloader_num_workers=0,
      seed=123,
)

print("TrainingArguments Experiment 2:")
print(f"  learning_rate: {training_args_exp2.learning_rate}")
print(f"  batch_size: {training_args_exp2.per_device_train_batch_size}")
print(f"  lr_scheduler: {training_args_exp2.lr_scheduler_type}")
print(f"  max_steps: {training_args_exp2.max_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments Experiment 2:
  learning_rate: 5e-05
  batch_size: 1
  lr_scheduler: SchedulerType.COSINE_WITH_RESTARTS
  max_steps: 800


In [ ]:
# Truncate the mapped text before SFTTrainer tokenization.
# This avoids Unsloth's runtime truncation path and keeps every sample within
# the 512-token context budget.
MAX_TRAIN_LENGTH = 384

def truncate_chatml(example):
    token_ids = tokenizer_exp2(
        example["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TRAIN_LENGTH,
    )["input_ids"]
    return {
        "text": tokenizer_exp2.decode(
            token_ids,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
    }

train_dataset_exp2 = train_dataset_mapped.map(
    truncate_chatml,
    num_proc=1,
    desc="Truncating Experiment 2 train dataset",
)
eval_dataset_exp2 = eval_dataset_mapped.map(
    truncate_chatml,
    num_proc=1,
    desc="Truncating Experiment 2 eval dataset",
)
eval_dataset_exp2 = eval_dataset_exp2.select(range(min(500, len(eval_dataset_exp2))))

trainer_exp2 = SFTTrainer(
    model=model_exp2,
    tokenizer=tokenizer_exp2,
    args=training_args_exp2,
    train_dataset=train_dataset_exp2,
    eval_dataset=eval_dataset_exp2,
    dataset_text_field="text",
    max_seq_length=MAX_TRAIN_LENGTH,
    dataset_num_proc=1,
    packing=False,
)

print("SFTTrainer Experiment 2 siap dengan truncation eksplisit.")

In [24]:
# Jalankan training — Experiment 2
print("MEMULAI TRAINING — EXPERIMENT 2")
print(f"  LoRA r=16, lora_alpha=32")
print(f"  lr=5e-5, batch_size=1, gradient_accumulation=8")
print(f"  max_steps=800")

print(f"Trainer batch size: {trainer_exp2.args.per_device_train_batch_size}")

trainer_exp2.train()

# Simpan log training
log_history_exp2 = trainer_exp2.state.log_history
print(f"\nTraining Experiment 2 selesai. Total log entries: {len(log_history_exp2)}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


MEMULAI TRAINING — EXPERIMENT 2
  LoRA r=16, lora_alpha=32
  lr=5e-5, batch_size=1, gradient_accumulation=8
  max_steps=800
Trainer batch size: 1


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,972 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Input IDs of shape torch.Size([1, 676]) with length 676 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <function cross_entropy at 0x7fded154aac0>(*(), **{'input': GradTrackingTensor(lvl=1, value=
    FakeTensor(..., device='cuda:0', size=(s97, 152064))
), 'target': GradTrackingTensor(lvl=1, value=
    FakeTensor(..., device='cuda:0', size=(s7,), dtype=torch.int64)
), 'reduction': 'sum', 'ignore_index': -100, 'label_smoothing': 0.0}): got ValueError('Expected input batch_size (s97: hint = 128) to match target batch_size (s7: hint = 169).')

from user code:
   File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/fused_losses/cross_entropy_loss.py", line 329, in accumulate_chunk
    (chunk_loss, (unscaled_loss,)) = torch.func.grad_and_value(
  File "/usr/local/lib/python3.12/dist-packages/torch/_functorch/apis.py", line 449, in wrapper
    return eager_transforms.grad_and_value_impl(
  File "/usr/local/lib/python3.12/dist-packages/torch/_functorch/vmap.py", line 48, in fn
    return f(*args, **kwargs)
  F

## 10. Loss Curve Comparison

Membandingkan kurva training loss dan evaluation loss dari kedua eksperimen untuk menentukan hyperparameter mana yang menghasilkan training paling optimal tanpa overfitting.

In [ ]:
# Ekstrak training dan evaluation loss
def extract_losses(log_history):
    """Ekstrak training loss dan eval loss dari log history."""
    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []

    for entry in log_history:
        if "loss" in entry and "eval_loss" not in entry:
            train_steps.append(entry.get("step", 0))
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry.get("step", 0))
            eval_losses.append(entry["eval_loss"])

    return train_steps, train_losses, eval_steps, eval_losses

t1_steps, t1_losses, e1_steps, e1_losses = extract_losses(log_history_exp1)
t2_steps, t2_losses, e2_steps, e2_losses = extract_losses(log_history_exp2)

print(f"Experiment 1 — Training loss points: {len(t1_losses)}, Eval loss points: {len(e1_losses)}")
print(f"Experiment 2 — Training loss points: {len(t2_losses)}, Eval loss points: {len(e2_losses)}")

In [ ]:
# Plot perbandingan loss curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss Comparison
ax1 = axes[0]
ax1.plot(t1_steps, t1_losses, label="Exp 1 (lr=2e-4, r=16)", color="#2196F3", linewidth=1.5, alpha=0.8)
ax1.plot(t2_steps, t2_losses, label="Exp 2 (lr=5e-5, r=32)", color="#FF5722", linewidth=1.5, alpha=0.8)
ax1.set_xlabel("Steps")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss Comparison")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Evaluation Loss Comparison
ax2 = axes[1]
ax2.plot(e1_steps, e1_losses, label="Exp 1 (lr=2e-4, r=16)", color="#2196F3",
         marker="o", markersize=5, linewidth=1.5)
ax2.plot(e2_steps, e2_losses, label="Exp 2 (lr=5e-5, r=32)", color="#FF5722",
         marker="s", markersize=5, linewidth=1.5)
ax2.set_xlabel("Steps")
ax2.set_ylabel("Evaluation Loss")
ax2.set_title("Evaluation Loss Comparison")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Fine-Tuning Qwen2.5-7B-Instruct: Eksperimen Hyperparameter",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analisis: Pilih model terbaik berdasarkan eval loss terendah
min_e1 = min(e1_losses) if e1_losses else float("inf")
min_e2 = min(e2_losses) if e2_losses else float("inf")

print("=" * 60)
print("ANALISIS PERBANDINGAN")
print("=" * 60)
print(f"Experiment 1 (lr=2e-4, r=16):")
print(f"  Training loss terakhir: {t1_losses[-1]:.4f}" if t1_losses else "  N/A")
print(f"  Eval loss terendah:    {min_e1:.4f}" if e1_losses else "  N/A")
print()
print(f"Experiment 2 (lr=5e-5, r=32):")
print(f"  Training loss terakhir: {t2_losses[-1]:.4f}" if t2_losses else "  N/A")
print(f"  Eval loss terendah:    {min_e2:.4f}" if e2_losses else "  N/A")
print()

# Tentukan best model
if min_e1 < min_e2:
    print("KESIMPULAN: Experiment 1 menghasilkan eval loss lebih rendah.")
    print("Menggunakan model dari Experiment 1 untuk final merge.")
    BEST_MODEL = model_exp1
    BEST_TOKENIZER = tokenizer_exp1
else:
    print("KESIMPULAN: Experiment 2 menghasilkan eval loss lebih rendah.")
    print("Menggunakan model dari Experiment 2 untuk final merge.")
    BEST_MODEL = model_exp2
    BEST_TOKENIZER = tokenizer_exp2

## 11. Merge LoRA & Simpan Model

Merge adapters LoRA ke base model dalam format **merged_16bit** (FP16) lalu push ke HuggingFace Hub.

In [ ]:
# Merge LoRA adapters ke model dan simpan dalam 16-bit
# Gunakan merged_16bit = True sesuai ketentuan submission

print("Menyimpan model hasil fine-tuning (merged_16bit)...")

# Nama model untuk HuggingFace Hub
HF_USERNAME = "RayhanLup1n"
MODEL_NAME = "qwen2.5-7b-indonesian-legal-finetuned"

# Merge & save locally dalam 16-bit
BEST_MODEL.save_pretrained_merged(
    f"./{MODEL_NAME}",
    BEST_TOKENIZER,
    save_method="merged_16bit",  # [WAJIB] merged dalam 16-bit
)

print(f"Model disimpan di: ./{MODEL_NAME}/")

In [ ]:
# Push model ke HuggingFace Hub
print(f"Mengunggah model ke HuggingFace Hub: {HF_USERNAME}/{MODEL_NAME}")

# Push merged model (cara 1: via push_to_hub)
BEST_MODEL.push_to_hub_merged(
    f"{HF_USERNAME}/{MODEL_NAME}",
    BEST_TOKENIZER,
    save_method="merged_16bit",  # [WAJIB] merged dalam 16-bit
    token=True,  # Gunakan token dari huggingface-cli login
)

print(f"Model berhasil diunggah: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

## 12. Simpan Link HuggingFace

In [ ]:
# Simpan link HuggingFace ke file txt
hf_url = f"https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}"

with open("link_huggingface.txt", "w", encoding="utf-8") as f:
    f.write(hf_url)

print(f"Link disimpan ke link_huggingface.txt")
print(f"URL: {hf_url}")

## Kesimpulan

Notebook ini telah menyelesaikan seluruh kriteria:

**Basic (2 pts)** ✅
- [x] Mapping dataset Alpaca ke ChatML template dengan token spesial
- [x] QLoRA 4-bit + double quantization
- [x] LoRA pada Multi-Head Attention (q_proj, k_proj, v_proj, o_proj)
- [x] SFTTrainer minimal 800 steps (tanpa OOM)
- [x] Push model ke HuggingFace (merged_16bit)

**Skilled (3 pts)** ✅
- [x] Train/validation split + eval_dataset
- [x] evaluation_strategy="steps" + logging
- [x] Dua eksperimen hyperparameter + loss curve comparison

**Advanced (4 pts)** ✅
- [x] Semua ketentuan Skilled terpenuhi
- [x] Siap untuk dilanjutkan ke GRPO training di notebook berikutnya